# Lab: how a multi-agent framework works

You write the core of a **supervisor** framework in LangGraph, run it on real questions, and look at what each agent reads. Then you compare it with one agent alone and with **handoffs**.

**The task:** four questions about European countries. Each one needs a **search** (population, area) and a **calculation** (density, ratio, difference).

Sections:

* **1 and 2:** the facts, the tools, and one agent as the reference.
* **3:** build the supervisor (6 blanks `___`).
* **4 to 6:** watch the team, measure it, change what each worker sees.
* **7 and 8:** handoffs, then a bonus on checkers.

**Setup:** put `OPENROUTER_API_KEY=...` in a `.env` file at the repository root (see `.env.example`); never commit it. Run the cells in order. The whole lab costs a few cents.

In [ ]:
import lab
from lab import ___                     # the blank marker: replace every ___ you see below

lab.check_setup()

## 1. The facts and the tools

The agents can only use these figures: population on 1 January 2025 (Eurostat) and area (UN Statistics Division). Each country has one document, with an id.

In [ ]:
lab.show_facts()

Tools are plain Python functions. The model never runs them: it asks for a call, and the code runs it.

In [ ]:
print(lab.search("Belgium population"))
print(lab.calculator("11900100 / 30528"))

The four questions, with the right answers computed from the table. Each answer must also cite the ids of the documents it uses.

* Population density of Belgium: **389.8** people per km².
* How many times larger is Germany's area than Belgium's: **11.7** times.
* Denser, the Netherlands or Belgium: **the Netherlands** (434.3 against 389.8).
* How many more people in Spain than in Portugal: **38.3** million.

The questions are plain strings. Every run below receives this list.

In [ ]:
QUESTIONS = [
    "What is the population density of Belgium, in people per km²?",
    "How many times larger is Germany's area than Belgium's?",
    "Which country is more densely populated, the Netherlands or Belgium? Give both densities.",
    "How many more people live in Spain than in Portugal?",
]
QUESTION = QUESTIONS[0]                              # the question we follow step by step

## 2. One agent with two tools

**One agent** with both tools, in a loop: call the model, run the tools it asks for, repeat.

How to read the output of `lab.trace`:

* **The table:** one row per step. Click **show prompt** to read the exact prompt of a model call.
* **The chart:** one row per agent, in step order. A circle is a model call and its size is the tokens it read. A square is a tool call.

In [ ]:
lab.trace("single", QUESTION)

Now the four questions. This run is the reference for the rest of the lab.

In [ ]:
lab.run_single(QUESTIONS)

## 3. Build the supervisor

A supervisor framework has three pieces:

1. **A shared state:** the question, a `transcript` that every worker appends to, a `hops` counter.
2. **The supervisor node:** one model call names the next worker (`lab.ask_router`). The node then sends the graph there with `Command(goto=...)`, or stops.
3. **The worker nodes:** an agent loop with its **own brief and tools**. When done, a worker writes to the transcript and goes back to the supervisor.

The numbers are the six blanks of the next cell:

![The supervisor graph with the six blanks placed on it](figures/blanks.svg)

Replace the six `___`. Tool names are strings, for example `"search"`.

In [ ]:
import operator
from typing import Annotated, TypedDict
from langgraph.graph import END
from langgraph.types import Command

MAX_HOPS = 6                                         # at most 6 worker turns per question


class TeamState(TypedDict):                          # piece 1: the shared state
    question: str
    transcript: Annotated[list[str], ___]           # blank 1: the reducer that appends each worker's entry
    hops: int


def supervisor(state: TeamState) -> Command:         # piece 2: the supervisor node
    choice = lab.ask_router(state)                   # one model call: "researcher", "analyst", "writer" or "FINISH"
    if choice == "FINISH" or state["hops"] >= ___:  # blank 2: stop after this many hops, whatever happens
        return Command(goto=END)
    return Command(goto=___, update={"hops": state["hops"] + 1})      # blank 3: which node runs next?


def make_worker(name, brief, tools):                 # piece 3: one worker node
    def worker(state: TeamState) -> Command:
        text = lab.agent_loop(brief, lab.view(state), tools, name)    # an agent loop with its own brief and tools
        return Command(goto=___, update={"transcript": [f"[{name}] {text}"]})   # blank 4: report back to whom?
    return worker


WORKERS = {                                          # name: (brief, tools)
    "researcher": ("Find the figures with the search tool. Quote them with their document id. Do not compute.", [___]),   # blank 5
    "analyst": ("Compute what the question asks from the figures found so far, with the calculator. Reply with the result only.", [___]),   # blank 6
    "writer": ("Write the final answer in one sentence from the work so far. Cite the document id of each figure in square brackets.", []),
}

This check uses a fake model, so it costs nothing. It names the blank to fix. After a fix, re-run the cell above, this check, and the wiring cell below.

In [ ]:
lab.check_team(TeamState, supervisor, make_worker, WORKERS, MAX_HOPS)

Now wire the pieces into a graph: nodes, the edges they may take, a starting point.

In [ ]:
from langgraph.graph import START, StateGraph

builder = StateGraph(TeamState)
builder.add_node("supervisor", supervisor, destinations=(*WORKERS, END))
for name, (brief, tools) in WORKERS.items():
    builder.add_node(name, make_worker(name, brief, tools), destinations=("supervisor",))
builder.add_edge(START, "supervisor")
team = builder.compile()

lab.show_graph(team)

## 4. Watch the team

How the supervisor decides: this is its whole brief.

In [ ]:
print(lab.ROUTER)

The question of section 2, now answered by the team. In the chart, the line goes back to the supervisor after every worker: a star. Open the prompt of the first and of the last supervisor call: what grew?

In [ ]:
lab.trace(team, QUESTION)

**The same graph in LangGraph Studio.** In a terminal, from this folder:

```bash
uv run langgraph dev
```

Open the Studio URL it prints, pick the graph **`supervisor`** and submit `{"question": "What is the population density of Belgium, in people per km²?"}`. Click each node to see what it received and returned. Open a **new thread** for each question: a thread keeps its transcript and its hops. The graph **`handoffs`** is there too, for section 7.

## 5. The team against one agent

The four questions, then both runs side by side. A team's answer counts only if the writer wrote it.

In [ ]:
lab.run_team(team, QUESTIONS)
lab.compare()

Is the team more accurate? How many times more calls and tokens does it use? What does one correct answer cost?

If an answer is wrong, read its stored run: `lab.replay("supervisor team", i)`, with `i` the question index from 0. A replay uses no tokens and shows the same run, while `lab.trace` starts a new run and the failure may not come back. Did the supervisor call every worker?

In [ ]:
lab.replay("supervisor team", 2)

## 6. What does each worker see?

So far every worker reads the **whole transcript**. Now each worker reads **only the last entry**. The supervisor still reads everything.

The writer must cite document ids, and only the researcher's entry holds them.

In [ ]:
lab.run_team(team, QUESTIONS, label="workers see the last entry", sees="last")
lab.compare()

Prompt tokens by agent, the two views together:

In [ ]:
lab.tokens_by_agent("supervisor team", "workers see the last entry")

One stored run, to read the writer's prompt:

In [ ]:
lab.replay("workers see the last entry", 0)

Which column changed, **correct** or **cites sources**? Did the total tokens go down, and which agent reads the most?

## 7. The other design: handoffs

No supervisor. The **same three workers**, each with extra tools: `transfer_to_researcher`, `transfer_to_analyst`, `transfer_to_writer`. When an agent calls one, the graph moves the **whole conversation** to that agent: the history travels, the brief changes.

In [ ]:
handoffs = lab.build_handoffs(WORKERS, MAX_HOPS)
lab.show_graph(handoffs)
lab.trace(handoffs, QUESTION)

In [ ]:
lab.run_team(handoffs, QUESTIONS, label="handoffs")
lab.compare()

Compare the two charts of a run: who decides the next step? Which design makes fewer calls, and which one reads more tokens per call?

A failed run stops early and makes fewer calls, which also lowers the total: check which runs finished before you compare call counts.

If an agent replies without calling a transfer tool, the run ends there, and only the writer's reply counts as an answer. A row marked `(no answer: ...)` is such a run: open it with `lab.replay("handoffs", i)`.